# VJ146A2 Full-Year 2023 Selected-Settlement Pixel Close-Ups

This notebook extends the earlier Jan-Jun close-up workflow to the full-year VJ146A2 panel. It includes the original five selected events plus 12 additional large-settlement notebook candidates screened for previous-day, event-day, and next-day support.

The plots are deliberately **not settlement-aggregated**. They show raw 500 m VJ pixels in a close-up window, with the settlement polygon contour overlaid.


## How To Read The Panels

Each settlement has two visual outputs:

1. **Satellite context panel**: Esri World Imagery basemap with the settlement contour in cyan.
2. **Raw VJ pixel close-up**: a 2 x 3 panel around the event date.

For the 2 x 3 panel:

- Columns: day before, event day, day after.
- Top row: valid radiance bins for the 500 m VJ pixels.
- Bottom row: binary pixel class using `rad > 1.0`.
- Grey means invalid or missing pixels.

The added candidates require actual previous/event/next rows, minimum three-day coverage of `0.80`, and at least `180` retained full-year observations.


In [ ]:
from pathlib import Path
import os
import subprocess

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        helper = candidate / "validation" / "closeups" / "scripts" / "vj146a2_full_year_selected_settlement_pixel_closeups.R"
        paths_helper = candidate / "validation" / "scripts" / "validation_paths.R"
        if helper.exists() and paths_helper.exists():
            return candidate
    raise FileNotFoundError("Could not locate Reliability-Assessment repo root with validation helpers")

REPO_ROOT = find_repo_root()
HELPER = REPO_ROOT / "validation" / "closeups" / "scripts" / "vj146a2_full_year_selected_settlement_pixel_closeups.R"
OUT_DIR = REPO_ROOT / "validation" / "closeups" / "figures" / "generated" / "vj146a2_full_year_selected_settlement_pixel_closeups"

paths = {
    "metrics": OUT_DIR / "vj146a2_full_year_selected_settlement_event_window_metrics.csv",
    "analysis": OUT_DIR / "vj146a2_full_year_selected_settlement_analysis_metrics.csv",
    "locations": OUT_DIR / "vj146a2_full_year_selected_settlement_locations.csv",
}

EVENTS = [
    {"settlement_id": "30250", "event_date": "2023-03-10", "label": "Maluti-a-Phofung Local Municipality", "group": "original"},
    {"settlement_id": "76354", "event_date": "2023-05-25", "label": "Bushbuckridge", "group": "original"},
    {"settlement_id": "73644", "event_date": "2023-05-19", "label": "Polokwane Local Municipality", "group": "original"},
    {"settlement_id": "69888", "event_date": "2023-04-17", "label": "Nkomazi", "group": "original"},
    {"settlement_id": "61860", "event_date": "2023-01-23", "label": "Thembisile Hani Local Municipality", "group": "original"},
    {"settlement_id": "61851", "event_date": "2023-04-21", "label": "Thembisile Hani Local Municipality", "group": "additional notebook"},
    {"settlement_id": "48939", "event_date": "2023-09-13", "label": "Matjhabeng Local Municipality", "group": "additional notebook"},
    {"settlement_id": "70733", "event_date": "2023-07-12", "label": "Elias Motsoaledi Local Municipality", "group": "additional notebook"},
    {"settlement_id": "55160", "event_date": "2023-04-17", "label": "Moses Kotane Local Municipality", "group": "additional notebook"},
    {"settlement_id": "49044", "event_date": "2023-05-22", "label": "Moqhaka Local Municipality", "group": "additional notebook"},
    {"settlement_id": "70495", "event_date": "2023-01-18", "label": "Dr JS Moroka Local Municipality", "group": "additional notebook"},
    {"settlement_id": "52564", "event_date": "2023-01-22", "label": "Westonaria Local Municipality", "group": "additional notebook"},
    {"settlement_id": "51558", "event_date": "2023-07-12", "label": "Merafong City Local Municipality", "group": "additional notebook"},
    {"settlement_id": "46678", "event_date": "2023-04-24", "label": "Ga-Segonyana Local Municipality", "group": "additional notebook"},
    {"settlement_id": "63077", "event_date": "2023-10-06", "label": "Albert Luthuli Local Municipality", "group": "additional notebook"},
    {"settlement_id": "61930", "event_date": "2023-07-12", "label": "Emalahleni Local Municipality", "group": "additional notebook"},
    {"settlement_id": "70237", "event_date": "2023-01-22", "label": "Bela Bela Local Municipality", "group": "additional notebook"},
]


def satellite_path(settlement_id, event_date):
    return OUT_DIR / f"satellite_closeup_{settlement_id}_{event_date}.png"


def pixel_path(settlement_id, event_date):
    return OUT_DIR / f"vj146a2_pixel_closeup_{settlement_id}_{event_date}.png"


## Regenerate Figures

Leave `RUN_R_HELPER = True` to refresh the satellite panels and raw VJ close-ups. The helper fetches Esri World Imagery tiles the first time it runs and caches them under the output folder. If tile download fails, rerun with network access enabled or reuse an existing tile cache.


In [ ]:
RUN_R_HELPER = True

if RUN_R_HELPER:
    result = subprocess.run(
        ["Rscript", str(HELPER.relative_to(REPO_ROOT))],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)

missing = [str(p) for p in paths.values() if not p.exists()]
for event in EVENTS:
    missing.extend(
        str(p)
        for p in [satellite_path(event["settlement_id"], event["event_date"]), pixel_path(event["settlement_id"], event["event_date"])]
        if not p.exists()
    )
if missing:
    raise FileNotFoundError("Missing expected close-up outputs:\n" + "\n".join(missing))


## Event Metrics and Helpers

The event-window metric rows come from the full-year yearlykeep settlement-day panel. The images below are raw pixels; these rows are shown only to label the before/event/after dates and provide context.


In [ ]:
metrics = pd.read_csv(paths["metrics"], parse_dates=["date", "event_date", "local_overpass_date"])
analysis = pd.read_csv(paths["analysis"], parse_dates=["event_date", "first_panel_date", "last_panel_date"])
locations = pd.read_csv(paths["locations"], parse_dates=["event_date"])


def _fmt_value(value, digits=4):
    if pd.isna(value):
        return "NA"
    return f"{value:.{digits}f}"


def analysis_benchmark(settlement_id):
    row = analysis.loc[analysis["settlement_id"].astype(str).eq(str(settlement_id))]
    if row.empty:
        raise ValueError(f"No full-year metrics for settlement_id={settlement_id}")
    row = row.iloc[0]
    records = [
        ("available panel window", f"{row['first_panel_date'].date()} to {row['last_panel_date'].date()}"),
        ("days retained in panel", f"{int(row['n_days'])}"),
        ("days with p_lit", f"{int(row['n_days_with_p_lit'])}"),
        ("p_lit full-year mean", _fmt_value(row["analysis_p_lit_mean"])),
        ("p_lit full-year median", _fmt_value(row["analysis_p_lit_median"])),
        ("p_lit full-year min", _fmt_value(row["analysis_p_lit_min"])),
        ("coverage full-year mean", _fmt_value(row["analysis_coverage_mean"])),
        ("coverage full-year median", _fmt_value(row["analysis_coverage_median"])),
        ("daily mean rad full-year mean", _fmt_value(row["analysis_daily_mean_rad_mean"])),
        ("daily median rad full-year median", _fmt_value(row["analysis_daily_median_rad_median"])),
        ("strict-dark days", f"{int(row['analysis_strict_dark_count'])}"),
        ("mostly-dark days", f"{int(row['analysis_mostly_dark_count'])}"),
        ("selection reason", row["selection_reason"]),
    ]
    return pd.DataFrame(records, columns=["metric", "value"])


def settlement_location(settlement_id, event_date):
    row = locations.loc[
        locations["settlement_id"].astype(str).eq(str(settlement_id))
        & locations["event_date"].dt.strftime("%Y-%m-%d").eq(event_date)
    ]
    if row.empty:
        raise ValueError(f"No location metadata for settlement_id={settlement_id}, event_date={event_date}")
    cols = [
        "event_label",
        "event_class",
        "village_name",
        "admin_cgaz_1",
        "admin_cgaz_2",
        "population",
        "representative_lat",
        "representative_lon",
        "google_earth_search",
    ]
    return row[cols]


def event_window(settlement_id, event_date):
    rows = metrics.loc[
        metrics["settlement_id"].astype(str).eq(str(settlement_id))
        & metrics["event_date"].dt.strftime("%Y-%m-%d").eq(event_date)
    ].copy()
    cols = [
        "relative_day",
        "date",
        "local_overpass_date",
        "p_lit_sett",
        "coverage",
        "mean_rad_sett",
        "median_rad_sett",
        "mlr_mean_1_2am_primary",
        "shed_share_1_2am_primary",
    ]
    return rows[cols]


def show_event(settlement_id, event_date):
    display(Image(filename=str(satellite_path(settlement_id, event_date))))
    display(Image(filename=str(pixel_path(settlement_id, event_date))))
    display(settlement_location(settlement_id, event_date))
    display(event_window(settlement_id, event_date))
    display(analysis_benchmark(settlement_id))


## Original Priority Events

These are the five events from the earlier selected-city notebook.


## Maluti-a-Phofung Local Municipality, 2023-03-10

Strict-dark event. Largest strict-dark candidate above 100,000 population.


In [ ]:
show_event("30250", "2023-03-10")


## Bushbuckridge, 2023-05-25

Strict-dark event. Full coverage and relatively low national shed share, useful for artifact/local-outage checking.


In [ ]:
show_event("76354", "2023-05-25")


## Polokwane Local Municipality, 2023-05-19

Strict-dark event. Full coverage and a large drop from a high baseline.


In [ ]:
show_event("73644", "2023-05-19")


## Nkomazi, 2023-04-17

Mostly-dark event. Very large settlement with full coverage and high Eskom shed share.


In [ ]:
show_event("69888", "2023-04-17")


## Thembisile Hani Local Municipality, 2023-01-23

Mostly-dark event. Large settlement with full coverage and sharp drop from a high baseline.


In [ ]:
show_event("61860", "2023-01-23")


## Additional Notebook Candidates

These 12 large-settlement candidates were added after screening for previous-day and next-day support, minimum three-day coverage, and at least 180 retained full-year observations.


## Thembisile Hani Local Municipality, 2023-04-21

Strict-dark sharp dip with high three-day support: p_lit 0.989 -> 0.000 -> 0.892.


In [ ]:
show_event("61851", "2023-04-21")


## Matjhabeng Local Municipality, 2023-09-13

Strict-dark sharp dip with full three-day coverage: p_lit 0.900 -> 0.028 -> 0.909.


In [ ]:
show_event("48939", "2023-09-13")


## Elias Motsoaledi Local Municipality, 2023-07-12

Strict-dark sharp dip with full three-day coverage: p_lit 0.869 -> 0.014 -> 0.904.


In [ ]:
show_event("70733", "2023-07-12")


## Moses Kotane Local Municipality, 2023-04-17

Strict-dark sharp dip with full three-day coverage: p_lit 0.743 -> 0.037 -> 0.815.


In [ ]:
show_event("55160", "2023-04-17")


## Moqhaka Local Municipality, 2023-05-22

Mostly-dark sharp dip with full three-day coverage: p_lit 1.000 -> 0.085 -> 1.000.


In [ ]:
show_event("49044", "2023-05-22")


## Dr JS Moroka Local Municipality, 2023-01-18

Mostly-dark sharp dip with high three-day coverage: p_lit 0.929 -> 0.092 -> 0.998.


In [ ]:
show_event("70495", "2023-01-18")


## Westonaria Local Municipality, 2023-01-22

Strict-dark event with full three-day coverage: p_lit 0.594 -> 0.038 -> 0.600.


In [ ]:
show_event("52564", "2023-01-22")


## Merafong City Local Municipality, 2023-07-12

Mostly-dark sharp dip with full three-day coverage: p_lit 0.935 -> 0.130 -> 0.941.


In [ ]:
show_event("51558", "2023-07-12")


## Ga-Segonyana Local Municipality, 2023-04-24

Mostly-dark sharp dip with full three-day coverage: p_lit 0.794 -> 0.164 -> 0.902.


In [ ]:
show_event("46678", "2023-04-24")


## Albert Luthuli Local Municipality, 2023-10-06

Mostly-dark sharp dip with usable three-day coverage: p_lit 0.970 -> 0.142 -> 0.999.


In [ ]:
show_event("63077", "2023-10-06")


## Emalahleni Local Municipality, 2023-07-12

Mostly-dark sharp dip with usable three-day coverage: p_lit 0.711 -> 0.090 -> 0.875.


In [ ]:
show_event("61930", "2023-07-12")


## Bela Bela Local Municipality, 2023-01-22

Mostly-dark event with full three-day coverage: p_lit 0.969 -> 0.198 -> 0.490.


In [ ]:
show_event("70237", "2023-01-22")


## Working Interpretation

Use these panels as a candidate-screening tool, not as confirmed outage evidence. A visually useful event should show a spatially coherent event-day darkening inside the settlement contour, with neighboring-day data available and enough valid pixels to rule out obvious missing-data artifacts.

The added candidates are selected specifically because previous-day and next-day rows exist and pass the coverage screen. Still check the raw valid mask and settlement boundary before using any individual event as evidence.
